# `Reporting Excel automatisé en Python`
*Oscar JOSEPH--GENESLAY*
**Data :** SuperStore Dataset
Source : *[Kaggle - SuperStore Dataset par vivek468](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final?resource=download)*

# Importation des packages

In [711]:
import pandas as pd
from datetime import datetime
import openpyxl
from openpyxl.chart import BarChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import FORMULAE

# Importation des données

In [712]:
dataset = pd.read_csv(
    "https://minio.lab.sspcloud.fr/oscar04/Superstore/Superstore.csv",
    encoding="windows-1252"
)

dataset.head(3)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [713]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

# Reformatage des colonnes

In [714]:
# Retypage
dataset["Row ID"] = str(dataset["Row ID"])
dataset["Postal Code"] = str(dataset["Postal Code"])


# Transformation en format date
dataset["Order Date"] = pd.to_datetime(dataset["Order Date"], format='%d/%m/%Y', errors='coerce')
dataset["Ship Date"] = pd.to_datetime(dataset["Ship Date"], format='%d/%m/%Y', errors='coerce')

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   object        
 1   Order ID       9994 non-null   object        
 2   Order Date     4042 non-null   datetime64[ns]
 3   Ship Date      3898 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   object        
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

# Fichier du reporting

## Création du fichier Excel

In [715]:
with pd.ExcelWriter("../output/reporting.xlsx", engine="openpyxl") as writer:
    dataset.to_excel(writer, sheet_name="DATA", index=False)

print("✅ reporting.xlsx créé")

✅ reporting.xlsx créé


## Chargement du reporting en mémoire

In [716]:
# Charger le classeur (workbook) en mémoire
wb = openpyxl.load_workbook('../output/reporting.xlsx')

# Création d'une feuille visualisations
ws_visualisations = wb.create_sheet(title="VISUALISATIONS")

# Définir la feuille DATA en variable python
ws_data = wb["DATA"]

# wb est un objet Python — on ne peut pas afficher son contenu avec un simple print
print(type(wb))
print('Feuilles disponibles :', wb.sheetnames)

<class 'openpyxl.workbook.workbook.Workbook'>
Feuilles disponibles : ['DATA', 'VISUALISATIONS']


# Création des KPIs

### KPIs numériques

##### CA ; Profits ; Quantité

In [717]:
# Création fonction d'automatisation
def numerical_kpis(cell, colonne, cell_format):
    ws_visualisations[cell] = f"=SUM(DATA!{colonne}:{colonne})"
    ws_visualisations[cell].number_format = cell_format

In [718]:
# CA
numerical_kpis("B6", "R", "#,##0.00 $")

# Profit
numerical_kpis("F6", "U", "#,##0.00 $")

# Quantité
numerical_kpis("J6", "S", "#,##0")

##### Temps de livraison

La fonction de calcul de différence entre date existe dans openpyxl

In [719]:
"DATEDIF" in FORMULAE

True

Application de la formule datedif sur chaque ligne où c'est calculable

In [720]:
for row_cell in range(2, len(dataset)+2):
    cell_application = f"V{row_cell}"
    date_debut = f"C{row_cell}"
    date_fin = f"D{row_cell}"
    ws_data[cell_application] = f'=IF(OR({date_debut} = "", {date_fin} = ""), "", DATEDIF({date_debut}, {date_fin}, "D"))'

Création du KPI moyenne 

In [721]:
ws_visualisations["O6"] = "=ROUND(AVERAGE(DATA!V:V),1)"

### TOP10 des sous-catégories avec le plus de profits

1. Calcul des profits par sous catégories et création du top 10

La fonction UNIQUE n'existe pas dans openpyxl

In [722]:
"UNIQUE" in FORMULAE

False

Calcul du nombre de sous catégories

In [723]:
len_sous_cat = len(dataset["Sub-Category"].unique())+1
# Calcul du nombre de sous catégories
len_dict ={}
for col in ["Sub-Category"]:
    len_dict["len_sous_cat"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
len_dict

{'len_sous_cat': 18}

Application de la formule UNIQUE

In [724]:
formula = "=_xlfn.UNIQUE(DATA!P:P)"
ws_data['Z2'] = ArrayFormula(f"Z2:Z{len_dict['len_sous_cat']+1}", formula)

Calculer le profits par sous catégories

In [725]:
for num_cell in range(3, len_dict['len_sous_cat']+2):
    cell = "AA" + str(num_cell)
    ws_data[cell] = "=SUMIF(DATA!P:P,DATA!Z" + str(num_cell) + ",DATA!U:U)"

La fonction "SORT" n'existe pas dans openpyxl

In [726]:
"SORT" in FORMULAE

False

Application du tri du tableau de données

In [727]:
# Trier le tableau de données
formula = "=_xlfn.SORT(DATA!Z3:AA19,2,-1)"
ws_data['AC3'] = ArrayFormula("AC3:AD19", formula)

2. Création du graphique à barres

In [728]:
# Création du graphique à barres
chart_top10 = BarChart()
chart_top10.type = "col"
chart_top10.title = "TOP 10 du profit en fonction des sous-catégories"
chart_top10.style = 10
chart_top10.width = 18
chart_top10.height = 10

# Récupération des données de profits
data_ref = Reference(ws_data, min_col=30, min_row=3, max_row=12)

# Récupération des labels
cats_ref = Reference(ws_data, min_col=29, min_row=4, max_row=12)

# Appliquer les paramètres au graphique
chart_top10.add_data(data_ref, titles_from_data=True)
chart_top10.set_categories(cats_ref)

# Titres des axes
chart_top10.y_axis.title = 'Profits (en $)'
chart_top10.x_axis.title = 'Sous-catégories'

# Légende
chart_top10.legend = None

# Placement du graphique
ws_visualisations.add_chart(chart_top10, "B15")

# Création de l'aspect design

### Le titre

In [729]:
ws_visualisations["L1"] = "Performances économique"
ws_visualisations["L1"].font = Font(bold=True, size=28, color="185EB8")

### KPIs numériques

In [730]:
def design_numerical_kpis(cell, nom):
    ws_visualisations[cell] = nom
    ws_visualisations[cell].font = Font(bold=True, size=16)

In [731]:
# CA
design_numerical_kpis("B5", "Chiffre d'affaires")

# Profits
design_numerical_kpis("F5", "Profits")

# Quantités
design_numerical_kpis("J5", "Volume des ventes")

# Temps livraison
design_numerical_kpis("O5", "Temps de livraison moyen (en jours)")


# Export du reporting

In [732]:
wb.save("../output/reporting.xlsx")